# DPAD Classification Sweep

Runs `--phases classification` for all 8 DPAD e1000 configs (z-as-behavior and z-as-neural × 4 sessions).  
Edit `SESSIONS` / `MODES` below to split across multiple Kaggle jobs if needed (12h GPU limit).

**Dataset required:** `KAGGLE_USERNAME/dpad-data`  
Contains: `results/dpad/*/inference/`, `model_*.pkl`, YAML configs, and source code under `src/`.

In [ ]:
# ── 1. Install packages not pre-installed on Kaggle GPU image ────────────────
# --no-deps + --ignore-requires-python: TF already on Kaggle; avoids version
# check failures on Python 3.12. PSID 1.2.6 overrides DPAD's 1.2.5 pin.
import subprocess, sys


def pip(*args):
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", *args], capture_output=True, text=True
    )
    if result.returncode != 0:
        print("INSTALL FAILED:", " ".join(args))
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
    else:
        print("OK:", " ".join(args))


pip("--no-deps", "--ignore-requires-python", "DPAD==0.0.9")
pip("--no-deps", "--ignore-requires-python", "PSID==1.2.6")
pip("polars>=1.0.0")

In [ ]:
# ── 2. Configure which experiments to run ────────────────────────────────────
# Reduce SESSIONS or MODES and run as two separate Kaggle jobs if 12h limit hit.
SESSIONS = ["PDI1_S2", "PDI1_S4", "PDI4_S2", "PDI4_S3"]
MODES = ["z-as-behavior", "z-as-neural"]

In [ ]:
# ── 3. Paths and sys.path ─────────────────────────────────────────────────────
import sys, os
from pathlib import Path

INPUT = Path("/kaggle/input/dpad-data")
WORK = Path("/kaggle/working")

# Source code bundled in the dataset under src/
sys.path.insert(0, str(INPUT / "src"))

# Project root = /kaggle/working; YAML has results.project_root: '.'
os.chdir(str(WORK))

import tensorflow as tf
import DPAD, PSID

print("TF:  ", tf.__version__)
print("DPAD: imported OK")
print("PSID: imported OK")

In [ ]:
# ── 4. Symlink result data into working dir ───────────────────────────────────
# /kaggle/input is read-only; /kaggle/working is writable.
# Symlink inference/ and model files so the pipeline can read them via
# project_root='.'. The classification/ dir will be created fresh (not linked).

for src_variant in sorted(INPUT.glob("results/dpad/dpad_*_dbs_*")):
    dst_variant = WORK / "results" / "dpad" / src_variant.name
    dst_variant.mkdir(parents=True, exist_ok=True)
    for item in src_variant.iterdir():
        link = dst_variant / item.name
        if not link.exists():
            os.symlink(item, link)

# YAML configs
dst_setups = WORK / "training" / "setups" / "dpad_modal"
dst_setups.mkdir(parents=True, exist_ok=True)
for yaml_file in (INPUT / "training/setups/dpad_modal").glob("*.yaml"):
    link = dst_setups / yaml_file.name
    if not link.exists():
        os.symlink(yaml_file, link)

(WORK / "logs" / "dpad").mkdir(parents=True, exist_ok=True)

variant_count = len(list((WORK / "results" / "dpad").iterdir()))
print(f"Symlinked {variant_count} variant dirs")

In [ ]:
# ── 5. Run classification for each config ────────────────────────────────────
import traceback

from utils.config import get_config
from utils.logger import setup_logging
from training.pipelines._base import FrameworkPipeline

YAML_DIR = WORK / "training" / "setups" / "dpad_modal"
errors = []

for mode in MODES:
    for session in SESSIONS:
        yaml_path = YAML_DIR / f"dpad_modal_{session}_{mode}.yaml"
        if not yaml_path.exists():
            print(f"SKIP (no yaml): {yaml_path.name}")
            continue

        print(f'\n{"="*60}')
        print(f"Running: {session} / {mode}")
        print(f'{"="*60}')

        try:
            config = get_config(str(yaml_path))
            log = setup_logging(
                f"dpad_{session}_{mode}",
                WORK / "logs" / "dpad" / f"{session}_{mode}.log",
            )
            FrameworkPipeline(config, log, phases=("classification",)).run()
            print(f"DONE: {session} / {mode}")
        except Exception as exc:
            print(f"ERROR: {session} / {mode}: {exc}")
            traceback.print_exc()
            errors.append((session, mode, str(exc)))

print(f"\nFinished. {len(errors)} errors.")
for s, m, e in errors:
    print(f"  FAILED {s}/{m}: {e}")

In [ ]:
# ── 6. List output sweep parquets ────────────────────────────────────────────
total_rows = 0
import polars as pl

for cls_dir in sorted(
    (WORK / "results" / "dpad").glob("dpad_*_dbs_both/classification")
):
    parquets = sorted(cls_dir.glob("sweep_*.parquet"))
    if not parquets:
        print(f"  MISSING: {cls_dir.parent.name}")
        continue
    # Show final sweep file (largest, has all rows)
    final = [
        p
        for p in parquets
        if not any(tag in p.name for tag in ["predictions", "forecast"])
    ]
    p = final[-1] if final else parquets[-1]
    df = pl.read_parquet(p)
    total_rows += len(df)
    size_kb = p.stat().st_size // 1024
    print(f"  {cls_dir.parent.name}: {len(df)} rows, {size_kb} KB -> {p.name}")

print(f"\nTotal rows across all configs: {total_rows}")